# DATA09 BK vs Other 特征重要性排序、降维与 6:4 测试验证

本 notebook 读取 6 类已经计算完成的 DATA09 特征 CSV：FL00、FL05、BK00、QJ00、BK05、QJ05。原始 `label` 保留为 6 个来源类别，用于样本数量稳定性和分布审计；建模目标单独构造为 `target_label`，只做二分类：`BK`（BK00、BK05）对 `Other`（FL00、FL05、QJ00、QJ05）。

本程序不重新计算原始信号特征，只负责合并、样本数量稳定性分析、源文件组级 6:4 划分、两套独立算法的特征重要性排序、Top-K 降维测试和可视化。

## 1. 环境初始化

导入依赖、设置绘图风格、定位项目根目录，并为本次分析创建带时间戳的输出目录。

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

# 优先向上查找 .git 来定位项目根目录，避免从 notebooks 目录启动时把结果写到 notebooks/outputs。
workspace = Path.cwd().resolve()
for candidate in [workspace, *workspace.parents]:
    if (candidate / '.git').exists():
        workspace = candidate
        break

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = workspace / 'outputs' / f'DATA09_feature_importance_from_csv_{RUN_TIMESTAMP}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'项目根目录: {workspace}')
print(f'运行时间戳: {RUN_TIMESTAMP}')
print(f'输出目录: {OUTPUT_ROOT}')


## 2. 输入配置

`FEATURE_INPUTS` 使用“目录 + 显式标签”配置。程序会递归读取每个目录下的 `features*.csv`，自动跳过日志 CSV 和重复路径。`label` 保留 6 类来源标签；`target_label` 固定为 `BK` 与 `Other`，用于训练、特征排序、降维和测试。

In [ ]:
# =========================
# 输入配置
# =========================

# 每个配置项对应一个已完成特征提取的数据目录。
# 程序会递归查找目录下的 features*.csv，并用这里配置的 label 覆盖/规范化来源标签。
# FEATURE_INPUTS = [
#     {
#         'label': 'FL00',  # flow v0 样本，二分类时归为 Other
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\DATA09_v0-flow_features'),
#     },
#     {
#         'label': 'FL05',  # flow v0.5 样本，二分类时归为 Other
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\DATA09_v0.5-flow_features'),
#     },
#     {
#         'label': 'BK00',  # BK v0 样本，二分类时归为 BK
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-bk_features_BK00'),
#     },
#     {
#         'label': 'QJ00',  # QJ v0 样本，二分类时归为 Other
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-qj_features_QJ00'),
#     },
#     {
#         'label': 'BK05',  # BK v0.5 样本，二分类时归为 BK
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-bk_features_BK05'),
#     },
#     {
#         'label': 'QJ05',  # QJ v0.5 样本，二分类时归为 Other
#         'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-qj_features_QJ05'),
#     },
# ]

# # BK 来源标签集合。只有这些来源标签会映射为二分类目标 BK。
# BK_SOURCE_LABELS = {'BK00', 'BK05'}

# # 6 个来源标签到二分类目标的映射。label 保留来源类别，target_label 用于建模。
# TARGET_LABEL_BY_SOURCE_LABEL = {
#     'BK00': 'BK',
#     'BK05': 'BK',
#     'FL00': 'Other',
#     'FL05': 'Other',
#     'QJ00': 'Other',
#     'QJ05': 'Other',
# }

FEATURE_INPUTS = [
    {
        'label': 'FL00',  # flow v0 样本，二分类时归为 Other
        'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\DATA09_v0-flow_features'),
    },
    # {
    #     'label': 'FL05',  # flow v0.5 样本，二分类时归为 Other
    #     'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\DATA09_v0.5-flow_features'),
    # },
    {
        'label': 'BK00',  # BK v0 样本，二分类时归为 BK
        'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-bk_features_BK00'),
    },
    {
        'label': 'QJ00',  # QJ v0 样本，二分类时归为 Other
        'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-qj_features_QJ00'),
    },
    # {
    #     'label': 'BK05',  # BK v0.5 样本，二分类时归为 BK
    #     'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-bk_features_BK05'),
    # },
    # {
    #     'label': 'QJ05',  # QJ v0.5 样本，二分类时归为 Other
    #     'path': Path(r'E:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-qj_features_QJ05'),
    # },
]

# BK 来源标签集合。只有这些来源标签会映射为二分类目标 BK。
BK_SOURCE_LABELS = {'BK00', 'BK05'}

# 6 个来源标签到二分类目标的映射。label 保留来源类别，target_label 用于建模。
TARGET_LABEL_BY_SOURCE_LABEL = {
    'BK00': 'BK',
    # 'BK05': 'BK',
    'FL00': 'Other',
    # 'FL05': 'Other',
    'QJ00': 'Other',
    # 'QJ05': 'Other',
}

# 全局随机种子。固定后，同一份输入数据的源文件组划分和模型随机性可复现。
RANDOM_STATE = 42

# 源文件组级测试集比例。这里是 60% 训练、40% 测试。
TEST_SIZE = 0.40

# 图表和表格默认展示的 Top 特征数量。
TOP_N = 30

# RandomForest 树数量。越大越稳定，但运行越慢。
N_ESTIMATORS = 500

# 是否额外做 holdout 测试集 permutation audit。注意该结果只审计，不参与训练划分。
RUN_PERMUTATION_IMPORTANCE = True

# permutation importance 的重复次数。越大越稳定，但越慢。
PERMUTATION_REPEATS = 3

# 训练集内部交叉验证折数，用于特征重要性稳定性估计。
CV_FOLDS = 3

# 算法1 RandomForest 的多随机种子稳定性配置。
RF_STABILITY_SEEDS = [11, 23, 37]

# 算法2 L1 LogisticRegression 的稳定性配置。为控制耗时，默认只跑 1 个种子。
LR_STABILITY_SEEDS = [101]

# 算法1中进入 CV permutation importance 的候选特征上限。
PERMUTATION_CANDIDATE_LIMIT = 80

# 测试集 permutation audit 只审计前 N 个候选特征。
TEST_PERMUTATION_TOP_N = 30

# Top-K 降维测试使用的特征数量列表。
TOPK_COMPARE_VALUES = [5, 10, 20, 30, 50, 80]

# 高相关特征判定阈值，使用 Spearman 相关系数绝对值。
CORRELATION_THRESHOLD = 0.95

# 近零方差特征判定阈值。
NEAR_ZERO_STD_THRESHOLD = 1e-12

# 输入 CSV 表头列数低于该值时，认为不是完整特征表并跳过。
MIN_FEATURE_CSV_COLUMNS = 100

# 算法2为了提速和缓解类别不均衡，使用“全部 BK + 抽样 Other”的训练子集。
# Other 抽样数量上限为 min(BK数量 * 该比例, ALGORITHM2_MAX_OTHER_ROWS)。
ALGORITHM2_OTHER_TO_BK_RATIO = 5
ALGORITHM2_MAX_OTHER_ROWS = 5000

# 样本数量稳定性分析：尝试的子采样样本量。
STABILITY_SAMPLE_SIZES = [5, 10, 20, 30, 50, 80, 100, 150, 200, 300, 500, 800, 1000, 1500, 2000, 3000, 5000]

# 每个样本量重复抽样次数。越大曲线越平滑，但越慢。
STABILITY_REPEATS = 20

# 稳定性判定阈值：子样本特征均值向量与全量均值向量的相对 L2 误差。
STABILITY_RELATIVE_ERROR_THRESHOLD = 0.10

# 至少连续多少个样本量点低于阈值，才认为该类特征趋于稳定。
STABILITY_CONSECUTIVE_POINTS = 2

print(f'目标训练集比例: {1 - TEST_SIZE:.0%}')
print(f'目标测试集比例: {TEST_SIZE:.0%}')
print('已配置的特征输入目录:')
for item in FEATURE_INPUTS:
    print(f"  {item['label']}: {item['path']}")

## 3. 工具函数

定义目录展开、CSV 加载、6 类来源标签规范化、BK/Other 二分类目标构造、共同数值特征筛选、样本数量稳定性分析、源文件分组划分和两套模型辅助函数。

In [ ]:
# =========================
# Helper functions
# =========================

META_COLUMNS = {
    'source_file_name', 'source_file_path', 'source_format',
    'source_group_name', 'source_channel_name', 'source_detail',
    'label', 'sample_type', 'target_label', 'window_mode',
    'window_id', 'window_start_index', 'window_end_index',
    'window_length_samples', 'window_step_samples', 'window_duration_s', 'window_start_offset_s',
    'window_start_datetime', 'window_start_ms', 'window_end_ms', 'window_n_samples',
    'sample_rate_hz', 'original_sample_rate_hz', 'source_n_samples', 'source_duration_s',
    'starttime_raw', 'arrival_time_raw', 'arrival_time', 'starttime', 'channel_index',
    'source_file_index_in_label', 'output_part', 'split', 'sample_id',
}


def normalize_source_label(value: object, fallback_label: str) -> str:
    if pd.isna(value) or str(value).strip() == '':
        return fallback_label
    text = str(value).strip()
    lower = text.lower()
    if lower in {'fl00', 'flow00', 'v0-flow', '0829-flow-v0', 'flow-v0', 'flow'}:
        return 'FL00' if fallback_label == 'FL00' else fallback_label
    if lower in {'fl05', 'flow05', 'v0.5-flow', 'v05-flow', 'flow-v05', 'flow-v0.5'}:
        return 'FL05'
    if lower in {'bk', 'bk00', 'v0-bk'}:
        return 'BK00'
    if lower in {'bk05', 'v05-bk', 'v0.5-bk'}:
        return 'BK05'
    if lower in {'qj', 'qj00', 'v0-qj'}:
        return 'QJ00'
    if lower in {'qj05', 'v05-qj', 'v0.5-qj'}:
        return 'QJ05'
    if text in TARGET_LABEL_BY_SOURCE_LABEL:
        return text
    return fallback_label


def make_target_label(source_label: str) -> str:
    if source_label not in TARGET_LABEL_BY_SOURCE_LABEL:
        raise ValueError(f'二分类目标不支持该来源标签: {source_label}')
    return TARGET_LABEL_BY_SOURCE_LABEL[source_label]


def csv_header_column_count(path: Path) -> int:
    with path.open('r', encoding='utf-8-sig', errors='replace') as f:
        header = f.readline().rstrip('\n\r')
    if not header:
        return 0
    return header.count(',') + 1


def expand_feature_inputs(inputs: list[dict]) -> list[dict]:
    rows = []
    seen = set()
    skipped_rows = []
    for item in inputs:
        label = str(item['label'])
        root = Path(item['path'])
        if not root.exists():
            raise FileNotFoundError(root)
        if root.is_file():
            candidates = [root]
        else:
            candidates = sorted(
                p for p in root.rglob('*.csv')
                if p.name.lower().startswith('features') and 'log' not in p.name.lower()
            )
        if not candidates:
            raise FileNotFoundError(f'No features*.csv found for {label}: {root}')
        for path in candidates:
            resolved = str(path.resolve())
            if resolved in seen:
                print(f'[警告] 已跳过重复输入: {path}')
                continue
            seen.add(resolved)
            header_cols = csv_header_column_count(path)
            if header_cols < MIN_FEATURE_CSV_COLUMNS:
                skipped_rows.append({
                    'label': label,
                    'path': str(path),
                    'header_columns': header_cols,
                    'reason': f'header column count < MIN_FEATURE_CSV_COLUMNS ({MIN_FEATURE_CSV_COLUMNS})',
                })
                print(f'[警告] 已跳过格式异常或非特征CSV: {path} (表头列数={header_cols})')
                continue
            rows.append({'label': label, 'path': path, 'header_columns': header_cols})
    if skipped_rows:
        skipped = pd.DataFrame(skipped_rows)
        display(skipped)
    if not rows:
        raise ValueError('表头列数过滤后没有剩余的有效特征 CSV。')
    return rows


def load_one_feature_csv(path: Path, configured_label: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path)
    except pd.errors.ParserError as exc:
        raise pd.errors.ParserError(f'{path}: {exc}') from exc
    df['feature_csv_path'] = str(path)
    df['feature_csv_name'] = path.name
    if 'label' in df.columns:
        raw_label = df['label']
    elif 'sample_type' in df.columns:
        raw_label = df['sample_type']
    else:
        raw_label = pd.Series([configured_label] * len(df), index=df.index)
    df['label'] = [normalize_source_label(v, configured_label) for v in raw_label]
    df['target_label'] = df['label'].map(make_target_label)
    if 'source_file_name' not in df.columns:
        df['source_file_name'] = path.stem
    df['sample_id'] = df['source_file_name'].astype(str)
    if 'window_mode' in df.columns:
        df['sample_id'] = df['sample_id'] + '|' + df['window_mode'].astype(str)
    elif 'window_id' in df.columns:
        df['sample_id'] = df['sample_id'] + '|window_' + df['window_id'].astype(str)
    return df


def find_feature_columns(frames: list[pd.DataFrame]) -> list[str]:
    common = set(frames[0].columns)
    for df in frames[1:]:
        common &= set(df.columns)
    candidates = [c for c in frames[0].columns if c in common and c not in META_COLUMNS and not c.startswith('feature_csv_')]
    numeric_cols = []
    for col in candidates:
        ok = True
        for df in frames:
            s = pd.to_numeric(df[col], errors='coerce')
            if s.notna().sum() == 0:
                ok = False
                break
        if ok:
            numeric_cols.append(col)
    return numeric_cols


def build_source_group_table(frame: pd.DataFrame, label_col: str = 'target_label') -> pd.DataFrame:
    group_table = (
        frame.assign(_label=frame[label_col].astype(str), _source=frame['source_file_name'].astype(str))
        .groupby(['_label', '_source'], dropna=False)
        .agg(
            rows=(label_col, 'size'),
            source_labels=('label', lambda s: ', '.join(sorted(set(map(str, s))))),
            feature_csvs=('feature_csv_name', lambda s: ', '.join(sorted(set(map(str, s))))),
        )
        .reset_index()
        .rename(columns={'_label': label_col, '_source': 'source_file_name'})
        .sort_values([label_col, 'source_file_name'])
        .reset_index(drop=True)
    )
    return group_table


def split_by_source_group(frame: pd.DataFrame, test_size: float, random_state: int, label_col: str = 'target_label'):
    labels = sorted(frame[label_col].astype(str).unique())
    if len(labels) != 2:
        raise ValueError(f'二分类目标必须正好 2 类，当前 {label_col}={labels}')

    group_table = build_source_group_table(frame, label_col=label_col)
    mixed_groups = frame.groupby('source_file_name')[label_col].nunique()
    if (mixed_groups > 1).any():
        bad = mixed_groups[mixed_groups > 1].index.astype(str).tolist()[:20]
        raise ValueError(
            f'发现同一个 source_file_name 同时属于多个 {label_col}，不能安全做源文件组划分。'
            f'请先检查或重命名源文件以保证跨目标标签唯一。示例: {bad}'
        )

    label_source_counts = group_table.groupby(label_col)['source_file_name'].nunique()
    too_few = label_source_counts[label_source_counts < 2]
    if not too_few.empty:
        raise ValueError(
            f'以下 {label_col} 的源文件组少于 2 个，不能做源文件级 train/test 划分: {too_few.to_dict()}'
        )

    rng = np.random.default_rng(random_state)
    train_sources: set[str] = set()
    test_sources: set[str] = set()
    split_rows = []

    for label in labels:
        sources = group_table.loc[group_table[label_col] == label, 'source_file_name'].astype(str).to_numpy()
        shuffled = sources.copy()
        rng.shuffle(shuffled)
        n_test = int(round(len(shuffled) * test_size))
        n_test = min(max(1, n_test), len(shuffled) - 1)
        test_for_label = set(shuffled[:n_test])
        train_for_label = set(shuffled[n_test:])
        train_sources.update(train_for_label)
        test_sources.update(test_for_label)
        split_rows.append({
            label_col: label,
            'source_files_total': int(len(shuffled)),
            'source_files_train': int(len(train_for_label)),
            'source_files_test': int(len(test_for_label)),
            'actual_source_train_ratio': len(train_for_label) / len(shuffled),
            'actual_source_test_ratio': len(test_for_label) / len(shuffled),
        })

    train_idx = frame.index[frame['source_file_name'].astype(str).isin(train_sources)].to_numpy()
    test_idx = frame.index[frame['source_file_name'].astype(str).isin(test_sources)].to_numpy()
    source_split_summary = pd.DataFrame(split_rows)
    return train_idx, test_idx, group_table, source_split_summary


def make_rf_pipeline(random_state: int, n_estimators: int = N_ESTIMATORS) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rf', RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=random_state,
            class_weight='balanced_subsample',
            n_jobs=-1,
        )),
    ])


def make_lr_pipeline(random_state: int) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            solver='liblinear',
            l1_ratio=1.0,
            C=0.5,
            max_iter=1000,
            class_weight='balanced',
            random_state=random_state,
        )),
    ])


def make_algorithm2_balanced_train_subset(
    x_train: pd.DataFrame,
    y_train_array: np.ndarray,
    encoder: LabelEncoder,
    random_state: int,
) -> tuple[pd.DataFrame, np.ndarray]:
    bk_code = int(encoder.transform(['BK'])[0])
    bk_pos = np.flatnonzero(y_train_array == bk_code)
    other_pos = np.flatnonzero(y_train_array != bk_code)
    if len(bk_pos) == 0 or len(other_pos) == 0:
        return x_train, y_train_array
    rng = np.random.default_rng(random_state)
    n_other = min(len(other_pos), max(len(bk_pos) * ALGORITHM2_OTHER_TO_BK_RATIO, len(bk_pos)), ALGORITHM2_MAX_OTHER_ROWS)
    chosen_other = rng.choice(other_pos, size=n_other, replace=False)
    chosen = np.concatenate([bk_pos, chosen_other])
    rng.shuffle(chosen)
    return x_train.iloc[chosen], y_train_array[chosen]


def build_cv_splitter(frame: pd.DataFrame, train_mask: pd.Series, n_splits: int, random_state: int, label_col: str = 'target_label'):
    train_frame = frame.loc[train_mask].copy()
    y_train_text = train_frame[label_col].astype(str)
    groups_train = train_frame['source_file_name'].astype(str)
    min_class_count = int(y_train_text.value_counts().min())
    usable_splits = max(2, min(n_splits, min_class_count))
    group_label_counts = train_frame.groupby('source_file_name')[label_col].nunique()
    label_group_counts = train_frame.groupby(label_col)['source_file_name'].nunique()
    if (group_label_counts <= 1).all() and (label_group_counts >= usable_splits).all():
        splitter = StratifiedGroupKFold(n_splits=usable_splits, shuffle=True, random_state=random_state)
        return splitter, groups_train.to_numpy(), usable_splits, 'StratifiedGroupKFold'
    splitter = StratifiedKFold(n_splits=usable_splits, shuffle=True, random_state=random_state)
    return splitter, None, usable_splits, 'StratifiedKFold(row-level fallback)'


def high_correlation_summary(x_numeric: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    corr = x_numeric.corr(method='spearman').abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .rename('abs_spearman_corr')
        .reset_index()
        .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'})
        .query('abs_spearman_corr >= @threshold')
        .sort_values('abs_spearman_corr', ascending=False)
        .reset_index(drop=True)
    )
    if len(pairs):
        per_feature = pd.concat([pairs['feature_a'], pairs['feature_b']], ignore_index=True).value_counts().rename_axis('feature').reset_index(name='high_corr_partner_count')
    else:
        per_feature = pd.DataFrame({'feature': [], 'high_corr_partner_count': []})
    return pairs, per_feature


def compute_univariate_f_scores(x_train: pd.DataFrame, y_train_array: np.ndarray, features: list[str]) -> pd.DataFrame:
    imputed = SimpleImputer(strategy='median').fit_transform(x_train.loc[:, features])
    scores, p_values = f_classif(imputed, y_train_array)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    p_values = np.nan_to_num(p_values, nan=1.0, posinf=1.0, neginf=1.0)
    return pd.DataFrame({'feature': features, 'univariate_f_score': scores, 'univariate_f_pvalue': p_values})


def compute_mutual_info_scores(x_train: pd.DataFrame, y_train_array: np.ndarray, features: list[str], random_state: int) -> pd.DataFrame:
    imputed = SimpleImputer(strategy='median').fit_transform(x_train.loc[:, features])
    scores = mutual_info_classif(imputed, y_train_array, random_state=random_state)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.DataFrame({'feature': features, 'mutual_info_score': scores})


def choose_permutation_candidates(ranking_frame: pd.DataFrame, limit: int) -> list[str]:
    if limit <= 0 or limit >= len(ranking_frame):
        return ranking_frame['feature'].tolist()
    half = max(1, limit // 2)
    candidates = []
    candidates.extend(ranking_frame.sort_values('rf_importance_seed_mean', ascending=False)['feature'].head(half).tolist())
    candidates.extend(ranking_frame.sort_values('univariate_f_score', ascending=False)['feature'].head(limit - half).tolist())
    seen = set()
    unique = []
    for feat in candidates:
        if feat not in seen:
            seen.add(feat)
            unique.append(feat)
    if len(unique) < limit:
        ranked = ranking_frame.sort_values(['rank_fast_combined', 'feature'])['feature'].tolist()
        for feat in ranked:
            if feat not in seen:
                seen.add(feat)
                unique.append(feat)
            if len(unique) >= limit:
                break
    return unique[:limit]


def correlation_pruned_features(ranking_frame: pd.DataFrame, corr_pair_frame: pd.DataFrame, score_col: str, threshold: float) -> list[str]:
    high_corr = {}
    if corr_pair_frame is not None and len(corr_pair_frame):
        for row in corr_pair_frame.itertuples(index=False):
            if row.abs_spearman_corr >= threshold:
                high_corr.setdefault(row.feature_a, set()).add(row.feature_b)
                high_corr.setdefault(row.feature_b, set()).add(row.feature_a)
    selected = []
    selected_set = set()
    ordered = ranking_frame.sort_values(score_col, ascending=False)['feature'].tolist()
    for feat in ordered:
        if any(feat in high_corr.get(chosen, set()) for chosen in selected_set):
            continue
        selected.append(feat)
        selected_set.add(feat)
    return selected


def evaluate_model(name: str, model: Pipeline, x_train: pd.DataFrame, y_train_array: np.ndarray, x_test: pd.DataFrame, y_test_array: np.ndarray, encoder: LabelEncoder) -> dict:
    model.fit(x_train, y_train_array)
    pred = model.predict(x_test)
    metrics = {
        'model': name,
        'labels': encoder.classes_.tolist(),
        'feature_count': int(x_train.shape[1]),
        'train_rows': int(len(x_train)),
        'test_rows': int(len(x_test)),
        'accuracy': float(accuracy_score(y_test_array, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_array, pred)),
        'macro_f1': float(f1_score(y_test_array, pred, average='macro')),
        'weighted_f1': float(f1_score(y_test_array, pred, average='weighted')),
    }
    return {'model': model, 'prediction': pred, 'metrics': metrics}


def evaluate_topk_feature_sets(rankings: dict[str, dict], topk_values: list[int]) -> pd.DataFrame:
    rows = []
    for method, spec in rankings.items():
        ranked_features = [f for f in spec['features'] if f in feature_cols]
        model_factory = spec['model_factory']
        fit_x = spec.get('fit_x', X_train)
        fit_y = spec.get('fit_y', y_train)
        for k in topk_values:
            chosen = ranked_features[:min(k, len(ranked_features))]
            if not chosen:
                continue
            subset_model = model_factory(RANDOM_STATE)
            subset_model.fit(fit_x.loc[:, chosen], fit_y)
            pred = subset_model.predict(X_test.loc[:, chosen])
            rows.append({
                'method': method,
                'model': spec['model_name'],
                'top_k': len(chosen),
                'fit_rows': int(len(fit_x)),
                'accuracy': float(accuracy_score(y_test, pred)),
                'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
                'macro_f1': float(f1_score(y_test, pred, average='macro')),
            })
    return pd.DataFrame(rows)


def estimate_class_sample_stability(x_numeric: pd.DataFrame, labels: pd.Series, sample_sizes: list[int], repeats: int, random_state: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(random_state)
    imputer = SimpleImputer(strategy='median')
    x_imp = pd.DataFrame(imputer.fit_transform(x_numeric), columns=x_numeric.columns, index=x_numeric.index)
    center = x_imp.median(axis=0)
    scale = (x_imp.quantile(0.75, axis=0) - x_imp.quantile(0.25, axis=0)).replace(0, np.nan)
    scale = scale.fillna(x_imp.std(axis=0).replace(0, np.nan)).fillna(1.0)
    x_scaled = (x_imp - center) / scale

    rows = []
    summary_rows = []
    for label in sorted(labels.astype(str).unique()):
        idx = labels.astype(str) == label
        class_x = x_scaled.loc[idx]
        full_mean = class_x.mean(axis=0).to_numpy()
        denom = float(np.linalg.norm(full_mean))
        if denom <= 1e-12:
            denom = 1.0
        usable_sizes = [n for n in sample_sizes if n <= len(class_x)]
        for n in usable_sizes:
            errors = []
            for _ in range(repeats):
                sample_pos = rng.choice(len(class_x), size=n, replace=False)
                sample_mean = class_x.iloc[sample_pos].mean(axis=0).to_numpy()
                errors.append(float(np.linalg.norm(sample_mean - full_mean) / denom))
            rows.append({
                'label': label,
                'sample_size': int(n),
                'class_rows': int(len(class_x)),
                'relative_l2_error_mean': float(np.mean(errors)),
                'relative_l2_error_std': float(np.std(errors, ddof=1)) if len(errors) > 1 else 0.0,
                'relative_l2_error_p90': float(np.quantile(errors, 0.90)),
                'repeats': int(repeats),
            })

        label_rows = [r for r in rows if r['label'] == label]
        stable_n = None
        for pos, row in enumerate(label_rows):
            window = label_rows[pos:pos + STABILITY_CONSECUTIVE_POINTS]
            if len(window) == STABILITY_CONSECUTIVE_POINTS and all(w['relative_l2_error_mean'] <= STABILITY_RELATIVE_ERROR_THRESHOLD for w in window):
                stable_n = row['sample_size']
                break
        summary_rows.append({
            'label': label,
            'class_rows': int(len(class_x)),
            'tested_min_sample_size': int(min(usable_sizes)) if usable_sizes else np.nan,
            'tested_max_sample_size': int(max(usable_sizes)) if usable_sizes else np.nan,
            'stable_sample_size_estimate': stable_n,
            'stability_threshold_relative_l2_error': STABILITY_RELATIVE_ERROR_THRESHOLD,
            'consecutive_points_required': STABILITY_CONSECUTIVE_POINTS,
        })

    return pd.DataFrame(rows), pd.DataFrame(summary_rows)


print('工具函数已准备完成。')

## 4. 特征表读取与合并

读取 6 个输入目录下的特征 CSV，输出每个文件的行数、列数、来源类别、二分类目标和源文件数，并合并为统一特征表。

In [ ]:
# =========================
# Step 1: Load CSV files, deduplicate inputs, and merge rows
# =========================

feature_sources = expand_feature_inputs(FEATURE_INPUTS)
frames = []
file_summary_rows = []

for item in feature_sources:
    path = item['path']
    configured_label = item['label']
    df = load_one_feature_csv(path, configured_label)
    frames.append(df)
    file_summary_rows.append({
        'configured_label': configured_label,
        'file': str(path),
        'header_columns': item.get('header_columns'),
        'rows': len(df),
        'columns': len(df.columns),
        'labels': ', '.join(sorted(df['label'].astype(str).unique())),
        'target_labels': ', '.join(sorted(df['target_label'].astype(str).unique())),
        'source_files': df['source_file_name'].nunique(),
    })

file_summary = pd.DataFrame(file_summary_rows)
display(file_summary)

labels_found = sorted(set().union(*(set(df['label'].astype(str).unique()) for df in frames)))
target_labels_found = sorted(set().union(*(set(df['target_label'].astype(str).unique()) for df in frames)))
expected_labels = sorted(TARGET_LABEL_BY_SOURCE_LABEL)
print(f'已读取 CSV 数量: {len(frames)}')
print(f'发现的来源标签: {labels_found}')
print(f'发现的二分类目标标签: {target_labels_found}')
missing = sorted(set(expected_labels) - set(labels_found))
if missing:
    print(f'[警告] 缺少预期来源标签: {missing}')
if target_labels_found != ['BK', 'Other']:
    raise ValueError(f'二分类目标异常，应为 BK/Other，当前为 {target_labels_found}')

feature_cols = find_feature_columns(frames)
print(f'共同数值特征列数量: {len(feature_cols)}')
if not feature_cols:
    raise ValueError('所有输入 CSV 中没有找到共同数值特征列。')

combined = pd.concat(frames, ignore_index=True, sort=False)
combined_path = OUTPUT_ROOT / f'combined_features_{RUN_TIMESTAMP}.csv'
combined.to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f'已保存合并特征表: {combined_path}')
print(f'合并后表格形状: {combined.shape}')
display(combined[['label', 'target_label', 'source_file_name', 'sample_id', 'feature_csv_name']].head(10))

## 5. 数据分布、特征质量与样本数量稳定性

统计 6 个来源类别和 BK/Other 二分类目标的样本数、源文件数和特征质量。随后用重复子采样估计每个来源类别的特征均值向量何时趋于稳定：当某个样本量下的平均相对 L2 误差连续达到阈值以内，即记录为该类的稳定样本量估计。

In [ ]:
# =========================
# Step 2: Class distribution, feature quality checks, and sample stability
# =========================

label_counts = combined['label'].value_counts().rename_axis('label').reset_index(name='rows')
source_counts = combined.groupby('label')['source_file_name'].nunique().rename('source_files').reset_index()
summary_counts = label_counts.merge(source_counts, on='label', how='left').sort_values('label').reset_index(drop=True)
display(summary_counts)

target_counts = combined['target_label'].value_counts().rename_axis('target_label').reset_index(name='rows')
target_source_counts = combined.groupby('target_label')['source_file_name'].nunique().rename('source_files').reset_index()
target_summary_counts = target_counts.merge(target_source_counts, on='target_label', how='left').sort_values('target_label').reset_index(drop=True)
display(target_summary_counts)

X_all_numeric = combined.loc[:, feature_cols].apply(pd.to_numeric, errors='coerce')
feature_quality = pd.DataFrame({
    'feature': feature_cols,
    'missing_rate': X_all_numeric.isna().mean().to_numpy(),
    'valid_count': X_all_numeric.notna().sum().to_numpy(),
    'mean': X_all_numeric.mean().to_numpy(),
    'std': X_all_numeric.std().to_numpy(),
    'variance': X_all_numeric.var().to_numpy(),
    'unique_count': X_all_numeric.nunique(dropna=True).to_numpy(),
})
feature_quality['near_zero_variance'] = feature_quality['std'].fillna(0) <= NEAR_ZERO_STD_THRESHOLD

corr_pairs, corr_per_feature = high_correlation_summary(X_all_numeric, CORRELATION_THRESHOLD)
feature_quality = feature_quality.merge(corr_per_feature, on='feature', how='left')
feature_quality['high_corr_partner_count'] = feature_quality['high_corr_partner_count'].fillna(0).astype(int)
feature_quality = feature_quality.sort_values(
    ['missing_rate', 'near_zero_variance', 'high_corr_partner_count', 'variance'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

feature_quality_path = OUTPUT_ROOT / f'feature_quality_{RUN_TIMESTAMP}.csv'
feature_quality.to_csv(feature_quality_path, index=False, encoding='utf-8-sig')
print(f'已保存特征质量表: {feature_quality_path}')
print(f'近零方差特征数量: {int(feature_quality["near_zero_variance"].sum())}')
print(f'高相关特征对数量，|Spearman r| >= {CORRELATION_THRESHOLD}: {len(corr_pairs)}')
display(feature_quality.head(20))

if len(corr_pairs):
    corr_pairs_path = OUTPUT_ROOT / f'high_correlation_pairs_{RUN_TIMESTAMP}.csv'
    corr_pairs.to_csv(corr_pairs_path, index=False, encoding='utf-8-sig')
    print(f'已保存高相关特征对表: {corr_pairs_path}')
    display(corr_pairs.head(20))

print('正在估计各来源类别的样本数量稳定性...')
stability_curve, stability_summary = estimate_class_sample_stability(
    X_all_numeric,
    combined['label'],
    STABILITY_SAMPLE_SIZES,
    STABILITY_REPEATS,
    RANDOM_STATE,
)
stability_curve_path = OUTPUT_ROOT / f'sample_stability_curve_{RUN_TIMESTAMP}.csv'
stability_summary_path = OUTPUT_ROOT / f'sample_stability_summary_{RUN_TIMESTAMP}.csv'
stability_curve.to_csv(stability_curve_path, index=False, encoding='utf-8-sig')
stability_summary.to_csv(stability_summary_path, index=False, encoding='utf-8-sig')
print(f'已保存样本稳定性曲线表: {stability_curve_path}')
print(f'已保存样本稳定性汇总表: {stability_summary_path}')
display(stability_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(summary_counts['label'], summary_counts['rows'], color='#4c78a8')
axes[0].set_title('Rows per source label')
axes[0].set_xlabel('source label')
axes[0].set_ylabel('rows')
axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(summary_counts['label'], summary_counts['source_files'], color='#f58518')
axes[1].set_title('Source files per source label')
axes[1].set_xlabel('source label')
axes[1].set_ylabel('source files')
axes[1].tick_params(axis='x', rotation=30)
for label, grp in stability_curve.groupby('label'):
    axes[2].plot(grp['sample_size'], grp['relative_l2_error_mean'], marker='o', linewidth=1.6, label=label)
axes[2].axhline(STABILITY_RELATIVE_ERROR_THRESHOLD, color='black', linewidth=0.9, linestyle='--')
axes[2].set_xscale('log')
axes[2].set_title('Feature mean stability by source label')
axes[2].set_xlabel('sample size')
axes[2].set_ylabel('mean relative L2 error')
axes[2].legend(loc='best')
plt.tight_layout()
class_plot_path = OUTPUT_ROOT / f'class_distribution_and_stability_{RUN_TIMESTAMP}.png'
plt.savefig(class_plot_path)
plt.show()
print(f'已保存图像: {class_plot_path}')

## 6. BK/Other 二分类 6:4 训练测试划分

按 `target_label` 做源文件组级 60% 训练、40% 测试划分。BK、QJ 和 flow 的同一 `source_file_name` 可能派生多个窗口样本，这些派生样本具有相似性，因此同一源文件的全部窗口必须进入同一个集合，禁止按窗口行级划分。

In [ ]:
# =========================
# Step 3: Train/test split grouped by source file for BK vs Other
# =========================

train_idx, test_idx, source_group_table, source_split_summary = split_by_source_group(
    combined,
    TEST_SIZE,
    RANDOM_STATE,
    label_col='target_label',
)
combined['split'] = 'train'
combined.loc[test_idx, 'split'] = 'test'

source_group_table_path = OUTPUT_ROOT / f'source_group_table_binary_{RUN_TIMESTAMP}.csv'
source_group_table.to_csv(source_group_table_path, index=False, encoding='utf-8-sig')
source_split_summary_path = OUTPUT_ROOT / f'source_split_summary_binary_{RUN_TIMESTAMP}.csv'
source_split_summary.to_csv(source_split_summary_path, index=False, encoding='utf-8-sig')

print('源文件组统计（按 BK/Other 目标标签）：')
display(source_group_table.groupby('target_label').agg(
    source_files=('source_file_name', 'nunique'),
    derived_window_rows=('rows', 'sum'),
    rows_per_source_min=('rows', 'min'),
    rows_per_source_max=('rows', 'max'),
    source_labels=('source_labels', lambda s: ', '.join(sorted(set(', '.join(s).split(', '))))),
).reset_index())

print('源文件组划分结果（划分比例按源文件组计算）：')
display(source_split_summary)
print(f'已保存源文件组统计表: {source_group_table_path}')
print(f'已保存源文件组划分表: {source_split_summary_path}')

split_summary = combined.groupby(['split', 'target_label']).agg(
    rows=('target_label', 'size'),
    source_files=('source_file_name', 'nunique'),
).reset_index()
display(split_summary)

source_label_split_summary = combined.groupby(['split', 'label', 'target_label']).agg(
    rows=('label', 'size'),
    source_files=('source_file_name', 'nunique'),
).reset_index()
display(source_label_split_summary)

train_sources = set(combined.loc[combined['split'] == 'train', 'source_file_name'].astype(str))
test_sources = set(combined.loc[combined['split'] == 'test', 'source_file_name'].astype(str))
overlap_sources = train_sources & test_sources
print(f'Train source files: {len(train_sources)}')
print(f'Test source files:  {len(test_sources)}')
print(f'Overlapped source files between train/test: {len(overlap_sources)}')
if overlap_sources:
    raise RuntimeError(f'源文件泄漏到 train/test 两边: {sorted(overlap_sources)[:20]}')

split_path = OUTPUT_ROOT / f'combined_features_with_split_{RUN_TIMESTAMP}.csv'
combined.to_csv(split_path, index=False, encoding='utf-8-sig')
print(f'已保存带划分标记的特征表: {split_path}')

## 7. BK/Other 全特征基线测试

用全部共同数值特征分别训练 RandomForest 和 L1 LogisticRegression 两个基线模型，并在 40% 源文件组级 holdout 测试集上输出二分类指标、分类报告和混淆矩阵。

In [ ]:
# =========================
# Step 4: Full-feature binary baselines
# =========================

X = combined.loc[:, feature_cols].apply(pd.to_numeric, errors='coerce')
y_text = combined['target_label'].astype(str)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

train_mask = combined['split'] == 'train'
test_mask = combined['split'] == 'test'

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y[train_mask.to_numpy()]
y_test = y[test_mask.to_numpy()]

rf_result = evaluate_model('algorithm1_random_forest_full_features', make_rf_pipeline(RANDOM_STATE), X_train, y_train, X_test, y_test, label_encoder)
lr_result = evaluate_model('algorithm2_l1_logistic_regression_full_features', make_lr_pipeline(RANDOM_STATE), X_train, y_train, X_test, y_test, label_encoder)

model = rf_result['model']
y_pred = rf_result['prediction']
lr_model = lr_result['model']
lr_pred = lr_result['prediction']

baseline_metrics = pd.DataFrame([rf_result['metrics'], lr_result['metrics']])
baseline_metrics_path = OUTPUT_ROOT / f'binary_baseline_metrics_{RUN_TIMESTAMP}.csv'
baseline_metrics.to_csv(baseline_metrics_path, index=False, encoding='utf-8-sig')
print(f'已保存全特征基线指标: {baseline_metrics_path}')
display(baseline_metrics)

for name, pred in [
    ('algorithm1_random_forest_full_features', y_pred),
    ('algorithm2_l1_logistic_regression_full_features', lr_pred),
]:
    report = classification_report(y_test, pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    report_path = OUTPUT_ROOT / f'classification_report_{name}_{RUN_TIMESTAMP}.csv'
    report_df.to_csv(report_path, encoding='utf-8-sig')
    print(f'已保存分类报告: {report_path}')
    display(report_df)

    cm = confusion_matrix(y_test, pred, labels=np.arange(len(label_encoder.classes_)))
    cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
    cm_path = OUTPUT_ROOT / f'confusion_matrix_{name}_{RUN_TIMESTAMP}.csv'
    cm_df.to_csv(cm_path, encoding='utf-8-sig')
    display(cm_df)
    print(f'已保存混淆矩阵: {cm_path}')

    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(np.arange(len(label_encoder.classes_)), labels=label_encoder.classes_)
    ax.set_yticks(np.arange(len(label_encoder.classes_)), labels=label_encoder.classes_)
    ax.set_xlabel('预测类别')
    ax.set_ylabel('真实类别')
    bacc = balanced_accuracy_score(y_test, pred)
    ax.set_title(f'{name}, balanced_acc={bacc:.3f}')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_plot_path = OUTPUT_ROOT / f'confusion_matrix_{name}_{RUN_TIMESTAMP}.png'
    plt.savefig(cm_plot_path)
    plt.show()
    print(f'已保存图像: {cm_plot_path}')

## 8. 使用算法1的特征重要性排序与降维、测试

算法1使用 RandomForest。排序证据包括：全特征 holdout 模型 impurity importance、多随机种子 RF 稳定性、训练集单变量 F 检验，以及训练集内分组交叉验证 permutation importance。降维测试使用 Top-K 特征子集和相关性去冗余 Top-K，在同一个 6:4 holdout 测试集上比较效果。

In [ ]:
# =========================
# Step 5: Algorithm 1, RandomForest feature ranking, dimensionality reduction, and test
# =========================

rf = model.named_steps['rf']
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance_holdout_model': rf.feature_importances_,
})

print('算法1：正在估计训练集上 RandomForest 多随机种子特征重要性稳定性...')
seed_importances = []
for seed in RF_STABILITY_SEEDS:
    seed_model = make_rf_pipeline(seed)
    seed_model.fit(X_train, y_train)
    seed_importances.append(seed_model.named_steps['rf'].feature_importances_)
seed_importances = np.vstack(seed_importances)
rf_stability = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance_seed_mean': seed_importances.mean(axis=0),
    'rf_importance_seed_std': seed_importances.std(axis=0, ddof=1) if len(RF_STABILITY_SEEDS) > 1 else np.zeros(len(feature_cols)),
})

univariate_scores = compute_univariate_f_scores(X_train, y_train, feature_cols)

fast_ranking = (
    rf_importance
    .merge(rf_stability, on='feature', how='left')
    .merge(univariate_scores, on='feature', how='left')
    .merge(feature_quality[['feature', 'missing_rate', 'near_zero_variance', 'high_corr_partner_count']], on='feature', how='left')
)
fast_ranking['rank_rf_stability'] = fast_ranking['rf_importance_seed_mean'].rank(ascending=False, method='min').astype(int)
fast_ranking['rank_univariate_f'] = fast_ranking['univariate_f_score'].rank(ascending=False, method='min').astype(int)
fast_ranking['rank_fast_combined'] = (fast_ranking['rank_rf_stability'] + fast_ranking['rank_univariate_f']) / 2.0

permutation_candidates = choose_permutation_candidates(fast_ranking, PERMUTATION_CANDIDATE_LIMIT)
print(
    f'算法1：正在对训练集候选特征运行交叉验证 permutation importance，候选特征数 {len(permutation_candidates)} '
    f'，总特征数 {len(feature_cols)}。'
)
cv_splitter, cv_groups, cv_splits, cv_name = build_cv_splitter(combined, train_mask, CV_FOLDS, RANDOM_STATE, label_col='target_label')
print(f'交叉验证划分器: {cv_name}, 折数={cv_splits}, 重复次数={PERMUTATION_REPEATS}')
cv_perm_rows = []
split_iter = cv_splitter.split(X_train, y_train) if cv_groups is None else cv_splitter.split(X_train, y_train, groups=cv_groups)

for fold, (fold_train_pos, fold_valid_pos) in enumerate(split_iter, start=1):
    fold_model = make_rf_pipeline(RANDOM_STATE + fold)
    fold_model.fit(X_train.iloc[fold_train_pos].loc[:, permutation_candidates], y_train[fold_train_pos])
    fold_pred = fold_model.predict(X_train.iloc[fold_valid_pos].loc[:, permutation_candidates])
    fold_score = balanced_accuracy_score(y_train[fold_valid_pos], fold_pred)
    perm = permutation_importance(
        fold_model,
        X_train.iloc[fold_valid_pos].loc[:, permutation_candidates],
        y_train[fold_valid_pos],
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE + fold,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )
    cv_perm_rows.append(pd.DataFrame({
        'fold': fold,
        'feature': permutation_candidates,
        'cv_permutation_importance': perm.importances_mean,
        'cv_permutation_importance_std_within_fold': perm.importances_std,
        'fold_balanced_accuracy': fold_score,
        'fold_valid_rows': len(fold_valid_pos),
    }))

cv_perm_long = pd.concat(cv_perm_rows, ignore_index=True)
cv_perm_long_path = OUTPUT_ROOT / f'algorithm1_rf_cv_permutation_importance_long_{RUN_TIMESTAMP}.csv'
cv_perm_long.to_csv(cv_perm_long_path, index=False, encoding='utf-8-sig')
print(f'已保存算法1 CV permutation 明细表: {cv_perm_long_path}')

cv_perm = (
    cv_perm_long.groupby('feature')
    .agg(
        cv_permutation_importance_mean=('cv_permutation_importance', 'mean'),
        cv_permutation_importance_std=('cv_permutation_importance', 'std'),
        cv_permutation_positive_fold_fraction=('cv_permutation_importance', lambda s: float((s > 0).mean())),
        cv_fold_balanced_accuracy_mean=('fold_balanced_accuracy', 'mean'),
    )
    .reset_index()
)
cv_perm['cv_permutation_importance_std'] = cv_perm['cv_permutation_importance_std'].fillna(0)

test_perm_df = pd.DataFrame({'feature': feature_cols})
if RUN_PERMUTATION_IMPORTANCE and len(permutation_candidates):
    audit_features = permutation_candidates[:min(TEST_PERMUTATION_TOP_N, len(permutation_candidates))]
    print(f'算法1：正在对 holdout 测试集前 {len(audit_features)} 个候选特征做 permutation 审计...')
    audit_model = make_rf_pipeline(RANDOM_STATE)
    audit_model.fit(X_train.loc[:, audit_features], y_train)
    test_perm = permutation_importance(
        audit_model,
        X_test.loc[:, audit_features],
        y_test,
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )
    test_perm_df = pd.DataFrame({
        'feature': audit_features,
        'test_permutation_importance_mean_audit_only': test_perm.importances_mean,
        'test_permutation_importance_std_audit_only': test_perm.importances_std,
    })

algorithm1_importance = (
    fast_ranking
    .merge(cv_perm, on='feature', how='left')
    .merge(test_perm_df, on='feature', how='left')
)
algorithm1_importance['in_permutation_candidate_set'] = algorithm1_importance['feature'].isin(permutation_candidates)
algorithm1_importance['cv_permutation_importance_mean'] = algorithm1_importance['cv_permutation_importance_mean'].fillna(0.0)
algorithm1_importance['cv_permutation_importance_std'] = algorithm1_importance['cv_permutation_importance_std'].fillna(0.0)
algorithm1_importance['cv_permutation_positive_fold_fraction'] = algorithm1_importance['cv_permutation_positive_fold_fraction'].fillna(0.0)
algorithm1_importance['stability_adjusted_score'] = algorithm1_importance['cv_permutation_importance_mean'] - algorithm1_importance['cv_permutation_importance_std']
algorithm1_importance['rank_cv_permutation'] = algorithm1_importance['cv_permutation_importance_mean'].rank(ascending=False, method='min').astype(int)
algorithm1_importance['rank_stability_adjusted'] = algorithm1_importance['stability_adjusted_score'].rank(ascending=False, method='min').astype(int)

algorithm1_all_cv_perm_zero = bool(np.isclose(algorithm1_importance['cv_permutation_importance_mean'].abs().max(), 0.0))
if algorithm1_all_cv_perm_zero:
    print('[警告] 算法1的 CV permutation importance 全为 0，排序回退到 RF/F-score 快速排序。')
    sort_cols = ['rank_fast_combined', 'rank_rf_stability', 'rank_univariate_f', 'feature']
else:
    sort_cols = ['rank_stability_adjusted', 'rank_cv_permutation', 'rank_fast_combined', 'feature']
algorithm1_importance = algorithm1_importance.sort_values(sort_cols).reset_index(drop=True)
algorithm1_importance.insert(0, 'importance_rank', np.arange(1, len(algorithm1_importance) + 1))

algorithm1_importance_path = OUTPUT_ROOT / f'algorithm1_rf_feature_importance_{RUN_TIMESTAMP}.csv'
algorithm1_importance.to_csv(algorithm1_importance_path, index=False, encoding='utf-8-sig')
print(f'已保存算法1特征重要性表: {algorithm1_importance_path}')
display(algorithm1_importance.head(TOP_N))

algorithm1_ranked_features = algorithm1_importance['feature'].tolist()
algorithm1_corr_pruned_features = correlation_pruned_features(algorithm1_importance, corr_pairs, 'rf_importance_seed_mean', CORRELATION_THRESHOLD)
algorithm1_topk_compare = evaluate_topk_feature_sets({
    'algorithm1_rf_cv_permutation_rank': {
        'features': algorithm1_ranked_features,
        'model_factory': make_rf_pipeline,
        'model_name': 'RandomForest',
    },
    'algorithm1_rf_correlation_pruned_rank': {
        'features': algorithm1_corr_pruned_features,
        'model_factory': make_rf_pipeline,
        'model_name': 'RandomForest',
    },
}, TOPK_COMPARE_VALUES)
algorithm1_topk_compare_path = OUTPUT_ROOT / f'algorithm1_rf_topk_dimensionality_test_{RUN_TIMESTAMP}.csv'
algorithm1_topk_compare.to_csv(algorithm1_topk_compare_path, index=False, encoding='utf-8-sig')
print(f'已保存算法1 Top-K 降维测试表: {algorithm1_topk_compare_path}')
display(algorithm1_topk_compare.sort_values(['balanced_accuracy', 'macro_f1', 'top_k'], ascending=[False, False, True]))

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for method, grp in algorithm1_topk_compare.groupby('method'):
    grp = grp.sort_values('top_k')
    ax.plot(grp['top_k'], grp['balanced_accuracy'], marker='o', linewidth=1.8, label=method)
ax.set_xlabel('Top-K selected features')
ax.set_ylabel('Holdout balanced accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Algorithm 1 RandomForest ranking / dimensionality reduction test')
ax.legend(loc='best')
plt.tight_layout()
algorithm1_compare_plot_path = OUTPUT_ROOT / f'algorithm1_rf_topk_dimensionality_test_{RUN_TIMESTAMP}.png'
plt.savefig(algorithm1_compare_plot_path)
plt.show()
print(f'已保存图像: {algorithm1_compare_plot_path}')

fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.24)))
plot_df = algorithm1_importance.head(TOP_N).iloc[::-1]
if algorithm1_all_cv_perm_zero:
    ax.barh(plot_df['feature'], plot_df['rf_importance_seed_mean'], color='#4c78a8', xerr=plot_df['rf_importance_seed_std'])
    ax.set_xlabel('RF impurity importance mean across seeds')
    ax.set_title(f'Algorithm 1 Top {TOP_N} features, RF fallback because CV permutation is all zero')
else:
    ax.barh(plot_df['feature'], plot_df['cv_permutation_importance_mean'], color='#4c78a8', xerr=plot_df['cv_permutation_importance_std'])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Train-CV permutation importance, balanced accuracy drop')
    ax.set_title(f'Algorithm 1 Top {TOP_N} features, train-CV stability adjusted')
plt.tight_layout()
algorithm1_importance_plot_path = OUTPUT_ROOT / f'algorithm1_rf_feature_importance_top{TOP_N}_{RUN_TIMESTAMP}.png'
plt.savefig(algorithm1_importance_plot_path)
plt.show()
print(f'已保存图像: {algorithm1_importance_plot_path}')

## 9. 使用算法2的特征重要性排序与降维、测试

算法2使用标准化后的 L1 LogisticRegression。L1 正则化会把一部分特征系数压到 0，天然适合稀疏特征选择；本节使用适合二分类的 `liblinear` 求解器，并在训练阶段使用“全 BK + 抽样 Other”的平衡训练子集降低类别极不均衡和样本量过大带来的耗时。排序使用绝对系数均值，并用互信息分数作为独立的非线性单变量参照。降维测试同样使用 Top-K 特征子集，在相同 6:4 holdout 测试集上验证。

注意：上一版为解决算法2长时间运行和 `scikit-learn 1.8` 的 `penalty/n_jobs` 警告，已将 L1 LogisticRegression 的求解器从较慢的 `saga` 调整为 `liblinear`，并去掉无效的 `n_jobs` 参数。因此算法2的全特征基线混淆矩阵可能与调整前略有差异；算法1 RandomForest 的基线算法没有因这次提速修改。


In [ ]:
# =========================
# Step 6: Algorithm 2, L1 LogisticRegression feature ranking, dimensionality reduction, and test
# =========================

section_start = perf_counter()
print('算法2：正在准备 L1 LogisticRegression 的平衡训练子集...')
X_train_algorithm2, y_train_algorithm2 = make_algorithm2_balanced_train_subset(
    X_train,
    y_train,
    label_encoder,
    RANDOM_STATE,
)
print(
    '算法2训练行数: '
    f'全量训练集={len(X_train)}, 平衡子集={len(X_train_algorithm2)}, '
    f'特征数={len(feature_cols)}, 类别计数={dict(zip(label_encoder.classes_, np.bincount(y_train_algorithm2, minlength=len(label_encoder.classes_))))}'
)

print('算法2：正在估计 L1 LogisticRegression 系数稳定性...')
step_start = perf_counter()
coef_rows = []
for seed in LR_STABILITY_SEEDS:
    seed_model = make_lr_pipeline(seed)
    seed_model.fit(X_train_algorithm2, y_train_algorithm2)
    coef = seed_model.named_steps['lr'].coef_
    if coef.shape[0] == 1:
        abs_coef = np.abs(coef[0])
    else:
        abs_coef = np.mean(np.abs(coef), axis=0)
    coef_rows.append(abs_coef)
    print(f'  seed={seed} 完成，耗时={perf_counter() - step_start:.1f}s')
coef_array = np.vstack(coef_rows)
lr_coef_stability = pd.DataFrame({
    'feature': feature_cols,
    'lr_abs_coef_seed_mean': coef_array.mean(axis=0),
    'lr_abs_coef_seed_std': coef_array.std(axis=0, ddof=1) if len(LR_STABILITY_SEEDS) > 1 else np.zeros(len(feature_cols)),
    'lr_nonzero_seed_fraction': (coef_array > 1e-12).mean(axis=0),
})

print('算法2：正在平衡子集上计算互信息分数...')
step_start = perf_counter()
mutual_info_scores = compute_mutual_info_scores(X_train_algorithm2, y_train_algorithm2, feature_cols, RANDOM_STATE)
print(f'  互信息计算完成，耗时={perf_counter() - step_start:.1f}s')

algorithm2_importance = (
    lr_coef_stability
    .merge(mutual_info_scores, on='feature', how='left')
    .merge(feature_quality[['feature', 'missing_rate', 'near_zero_variance', 'high_corr_partner_count']], on='feature', how='left')
)
algorithm2_importance['lr_stability_adjusted_score'] = algorithm2_importance['lr_abs_coef_seed_mean'] - algorithm2_importance['lr_abs_coef_seed_std']
algorithm2_importance['rank_l1_logreg_coef'] = algorithm2_importance['lr_abs_coef_seed_mean'].rank(ascending=False, method='min').astype(int)
algorithm2_importance['rank_l1_logreg_stability_adjusted'] = algorithm2_importance['lr_stability_adjusted_score'].rank(ascending=False, method='min').astype(int)
algorithm2_importance['rank_mutual_info'] = algorithm2_importance['mutual_info_score'].rank(ascending=False, method='min').astype(int)
algorithm2_importance['rank_algorithm2_combined'] = (
    algorithm2_importance['rank_l1_logreg_stability_adjusted'] + algorithm2_importance['rank_mutual_info']
) / 2.0
algorithm2_importance = algorithm2_importance.sort_values(
    ['rank_algorithm2_combined', 'rank_l1_logreg_stability_adjusted', 'rank_mutual_info', 'feature']
).reset_index(drop=True)
algorithm2_importance.insert(0, 'importance_rank', np.arange(1, len(algorithm2_importance) + 1))

algorithm2_importance_path = OUTPUT_ROOT / f'algorithm2_l1_logreg_feature_importance_{RUN_TIMESTAMP}.csv'
algorithm2_importance.to_csv(algorithm2_importance_path, index=False, encoding='utf-8-sig')
print(f'已保存算法2特征重要性表: {algorithm2_importance_path}')
display(algorithm2_importance.head(TOP_N))

algorithm2_ranked_features = algorithm2_importance['feature'].tolist()
algorithm2_nonzero_features = algorithm2_importance.loc[algorithm2_importance['lr_nonzero_seed_fraction'] > 0, 'feature'].tolist()
if not algorithm2_nonzero_features:
    print('[警告] 算法2 L1 模型没有选出非零系数，回退到综合排序。')
    algorithm2_nonzero_features = algorithm2_ranked_features
algorithm2_corr_pruned_features = correlation_pruned_features(algorithm2_importance, corr_pairs, 'lr_abs_coef_seed_mean', CORRELATION_THRESHOLD)

print('算法2：正在同一个 holdout 测试集上运行 Top-K 降维测试...')
step_start = perf_counter()
algorithm2_topk_compare = evaluate_topk_feature_sets({
    'algorithm2_l1_logreg_combined_rank': {
        'features': algorithm2_ranked_features,
        'model_factory': make_lr_pipeline,
        'model_name': 'L1 LogisticRegression',
        'fit_x': X_train_algorithm2,
        'fit_y': y_train_algorithm2,
    },
    'algorithm2_l1_logreg_nonzero_first': {
        'features': algorithm2_nonzero_features,
        'model_factory': make_lr_pipeline,
        'model_name': 'L1 LogisticRegression',
        'fit_x': X_train_algorithm2,
        'fit_y': y_train_algorithm2,
    },
    'algorithm2_l1_logreg_correlation_pruned_rank': {
        'features': algorithm2_corr_pruned_features,
        'model_factory': make_lr_pipeline,
        'model_name': 'L1 LogisticRegression',
        'fit_x': X_train_algorithm2,
        'fit_y': y_train_algorithm2,
    },
}, TOPK_COMPARE_VALUES)
print(f'  Top-K 测试完成，耗时={perf_counter() - step_start:.1f}s')
algorithm2_topk_compare_path = OUTPUT_ROOT / f'algorithm2_l1_logreg_topk_dimensionality_test_{RUN_TIMESTAMP}.csv'
algorithm2_topk_compare.to_csv(algorithm2_topk_compare_path, index=False, encoding='utf-8-sig')
print(f'已保存算法2 Top-K 降维测试表: {algorithm2_topk_compare_path}')
display(algorithm2_topk_compare.sort_values(['balanced_accuracy', 'macro_f1', 'top_k'], ascending=[False, False, True]))

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for method, grp in algorithm2_topk_compare.groupby('method'):
    grp = grp.sort_values('top_k')
    ax.plot(grp['top_k'], grp['balanced_accuracy'], marker='o', linewidth=1.8, label=method)
ax.set_xlabel('Top-K 选中特征数')
ax.set_ylabel('Holdout 平衡准确率')
ax.set_ylim(0, 1.05)
ax.set_title('Algorithm 2 L1 LogisticRegression ranking / dimensionality reduction test')
ax.legend(loc='best')
plt.tight_layout()
algorithm2_compare_plot_path = OUTPUT_ROOT / f'algorithm2_l1_logreg_topk_dimensionality_test_{RUN_TIMESTAMP}.png'
plt.savefig(algorithm2_compare_plot_path)
plt.show()
print(f'已保存图像: {algorithm2_compare_plot_path}')

fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.24)))
plot_df = algorithm2_importance.head(TOP_N).iloc[::-1]
ax.barh(plot_df['feature'], plot_df['lr_abs_coef_seed_mean'], color='#54a24b', xerr=plot_df['lr_abs_coef_seed_std'])
ax.set_xlabel('Mean absolute standardized coefficient across seeds')
ax.set_title(f'Algorithm 2 Top {TOP_N} features, L1 LogisticRegression')
plt.tight_layout()
algorithm2_importance_plot_path = OUTPUT_ROOT / f'algorithm2_l1_logreg_feature_importance_top{TOP_N}_{RUN_TIMESTAMP}.png'
plt.savefig(algorithm2_importance_plot_path)
plt.show()
print(f'已保存图像: {algorithm2_importance_plot_path}')
print(f'算法2章节总耗时: {perf_counter() - section_start:.1f}s')

## 10. 两种算法的降维测试汇总

汇总算法1和算法2的 Top-K 降维测试结果，便于直接比较两种独立排序方法在 BK/Other 二分类上的最小有效特征数量。

In [ ]:
# =========================
# Step 7: Combined algorithm comparison
# =========================

ranking_compare = pd.concat([algorithm1_topk_compare, algorithm2_topk_compare], ignore_index=True)
ranking_compare_path = OUTPUT_ROOT / f'feature_ranking_topk_comparison_two_algorithms_{RUN_TIMESTAMP}.csv'
ranking_compare.to_csv(ranking_compare_path, index=False, encoding='utf-8-sig')
print(f'已保存两种算法 Top-K 对比表: {ranking_compare_path}')
display(ranking_compare.sort_values(['balanced_accuracy', 'macro_f1', 'top_k'], ascending=[False, False, True]))

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for method, grp in ranking_compare.groupby('method'):
    grp = grp.sort_values('top_k')
    ax.plot(grp['top_k'], grp['balanced_accuracy'], marker='o', linewidth=1.6, label=method)
ax.set_xlabel('Top-K 选中特征数')
ax.set_ylabel('Holdout 平衡准确率')
ax.set_ylim(0, 1.05)
ax.set_title('Two independent algorithms: feature ranking and dimensionality reduction test')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
compare_plot_path = OUTPUT_ROOT / f'feature_ranking_topk_comparison_two_algorithms_{RUN_TIMESTAMP}.png'
plt.savefig(compare_plot_path)
plt.show()
print(f'已保存图像: {compare_plot_path}')

## 11. 错误样本与 Top 特征分布

列出测试集中 BK/Other 二分类错误样本，并分别绘制算法1、算法2 Top 特征在 6 个来源类别中的分布。

In [ ]:
# =========================
# Step 8: Misclassified samples and count-based top feature distributions
# =========================

test_rows = combined.loc[test_mask, ['label', 'target_label', 'source_file_name', 'sample_id', 'feature_csv_name']].copy()
test_rows['algorithm1_pred_target_label'] = label_encoder.inverse_transform(y_pred)
test_rows['algorithm2_pred_target_label'] = label_encoder.inverse_transform(lr_pred)
test_rows['algorithm1_correct'] = test_rows['target_label'].astype(str) == test_rows['algorithm1_pred_target_label'].astype(str)
test_rows['algorithm2_correct'] = test_rows['target_label'].astype(str) == test_rows['algorithm2_pred_target_label'].astype(str)
misclassified = test_rows.loc[~test_rows['algorithm1_correct'] | ~test_rows['algorithm2_correct']].copy()
misclassified_path = OUTPUT_ROOT / f'misclassified_samples_two_algorithms_{RUN_TIMESTAMP}.csv'
misclassified.to_csv(misclassified_path, index=False, encoding='utf-8-sig')
print(f'至少被一种算法误判的测试样本数: {len(misclassified)}')
print(f'已保存误判样本表: {misclassified_path}')
display(misclassified.head(50))

plot_feature_sets = {
    'algorithm1_rf': algorithm1_importance.head(min(6, len(algorithm1_importance)))['feature'].tolist(),
    'algorithm2_l1_logreg': algorithm2_importance.head(min(6, len(algorithm2_importance)))['feature'].tolist(),
}

for plot_name, plot_features in plot_feature_sets.items():
    if not plot_features:
        continue
    n = len(plot_features)
    fig, axes = plt.subplots(n, 1, figsize=(8, max(3, 2.4 * n)), sharex=False)
    if n == 1:
        axes = [axes]
    for ax, feat in zip(axes, plot_features):
        all_values = pd.to_numeric(combined[feat], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if all_values.empty:
            ax.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(feat)
            continue

        positive = all_values[all_values > 0]
        use_log = False
        if len(positive) == len(all_values):
            q01 = positive.quantile(0.01)
            q99 = positive.quantile(0.99)
            use_log = bool(q01 > 0 and q99 / q01 > 1e4)

        plotted_any = False
        for label in sorted(combined['label'].astype(str).unique()):
            raw = pd.to_numeric(
                combined.loc[combined['label'].astype(str) == label, feat],
                errors='coerce',
            ).replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty:
                continue
            values = np.log10(raw.to_numpy()) if use_log else raw.to_numpy()
            ax.hist(values, bins=30, alpha=0.45, density=False, label=f'{label} (n={len(raw)})')
            plotted_any = True

        if not plotted_any:
            ax.text(0.5, 0.5, 'no finite values to plot', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{plot_name}: {feat}' + (' [log10 scale]' if use_log else ''))
        ax.set_ylabel('count')
        ax.legend(loc='best')
    axes[-1].set_xlabel('feature value')
    plt.tight_layout()
    dist_plot_path = OUTPUT_ROOT / f'{plot_name}_top_feature_distributions_{RUN_TIMESTAMP}.png'
    plt.savefig(dist_plot_path)
    plt.show()
    print(f'已保存图像: {dist_plot_path}')

## 12. Top 特征分布的均衡可视化补充

上一节保留原始计数直方图，适合观察每类绝对样本量。本节新增归一化密度直方图和 ECDF：每个来源类别各自归一化，避免样本数差异主导图形高度，更适合比较 6 类信号在同一特征上的取值范围、峰值位置和分布偏移。

In [ ]:
# =========================
# Step 9: Balanced top-feature distribution plots
# =========================

for plot_name, plot_features_balanced in plot_feature_sets.items():
    if not plot_features_balanced:
        continue
    n = len(plot_features_balanced)
    fig, axes = plt.subplots(n, 2, figsize=(12, max(3.2, 2.6 * n)), sharex=False)
    if n == 1:
        axes = np.array([axes])

    for row_idx, feat in enumerate(plot_features_balanced):
        ax_hist = axes[row_idx, 0]
        ax_ecdf = axes[row_idx, 1]
        all_values = pd.to_numeric(combined[feat], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if all_values.empty:
            ax_hist.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax_hist.transAxes)
            ax_ecdf.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax_ecdf.transAxes)
            continue

        positive = all_values[all_values > 0]
        use_log = False
        if len(positive) == len(all_values):
            q01 = positive.quantile(0.01)
            q99 = positive.quantile(0.99)
            use_log = bool(q01 > 0 and q99 / q01 > 1e4)

        all_plot_values = np.log10(all_values.to_numpy()) if use_log else all_values.to_numpy()
        bins = np.histogram_bin_edges(all_plot_values, bins=30)

        for label in sorted(combined['label'].astype(str).unique()):
            raw = pd.to_numeric(
                combined.loc[combined['label'].astype(str) == label, feat],
                errors='coerce',
            ).replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty:
                continue
            values = np.log10(raw.to_numpy()) if use_log else raw.to_numpy()
            ax_hist.hist(values, bins=bins, alpha=0.35, density=True, label=f'{label} (n={len(raw)})')

            sorted_values = np.sort(values)
            ecdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
            ax_ecdf.step(sorted_values, ecdf, where='post', linewidth=1.8, label=f'{label} (n={len(raw)})')

        title_suffix = ' [log10 scale]' if use_log else ''
        ax_hist.set_title(f'{plot_name}: {feat} density{title_suffix}')
        ax_hist.set_ylabel('density')
        ax_hist.legend(loc='best')
        ax_ecdf.set_title(f'{plot_name}: {feat} ECDF{title_suffix}')
        ax_ecdf.set_ylabel('cumulative fraction')
        ax_ecdf.set_ylim(0, 1.02)
        ax_ecdf.legend(loc='best')

    axes[-1, 0].set_xlabel('feature value')
    axes[-1, 1].set_xlabel('feature value')
    plt.tight_layout()
    balanced_dist_plot_path = OUTPUT_ROOT / f'{plot_name}_top_feature_distributions_density_ecdf_{RUN_TIMESTAMP}.png'
    plt.savefig(balanced_dist_plot_path)
    plt.show()
    print(f'已保存图像: {balanced_dist_plot_path}')

## 13. 输出文件清单

In [ ]:
# =========================
# Step 10: Output artifact list
# =========================

artifacts = sorted(OUTPUT_ROOT.glob('*'))
print(f'输出文件目录 {OUTPUT_ROOT}:')
for path in artifacts:
    print(' ', path.name)